# Tutorial · Phân tích chứng khoán với TCInvest và SQL
**SpurLab Notebook · Tiếng Việt · Từ truy vấn đầu tiên đến một báo cáo thị trường**

Sau bài này, bạn có thể gắn global connection `tcinvest`, tìm bảng và tham số, viết SQL cell, kiểm tra dữ liệu, tính lợi suất/đường trung bình, đọc dòng vốn và so sánh 19 ngành. Thời gian thực hành gợi ý: **35–50 phút**.

Bài học dùng ngày chốt **07/10/2026** để đối chiếu với notebook phân tích trước. Đây là ví dụ có ngày chốt cố định, không tự nhận là báo cáo thời gian thực. Dữ liệu được lấy lại từ connector khi chạy các cell tải nguồn.

## Chuẩn bị khi mở bản từ GitHub

1. Làm theo [README của repository](../README.md#1-get-the-prerequisites): cài SpurLab, khởi động REST bridge và đăng nhập bằng tài khoản TCBS của bạn. Giữ bridge chạy trên cùng máy với kernel.
2. Trong trình nhập connection **REST/OpenAPI**, nhập `http://127.0.0.1:8788/openapi.json` hoặc [tcinvest.openapi.yaml](../tcinvest.openapi.yaml), đặt tên global connection là **`tcinvest`**, rồi gắn vào notebook này.
3. Bài học dùng **các cột từ OpenAPI**, ví dụ `tradingdate`, `seqtime`, `accvalue`. Manifest TOML tùy chọn có phép chiếu và tên cột khác; dùng connection OpenAPI cho bài này. Kiểm tra catalog nếu đã có cấu hình `tcinvest` cũ.
4. Mở file này trong **SpurLab** và chạy cell datasource setup mà ứng dụng sinh ra, rồi chạy các SQL cell theo thứ tự. Jupyter với Python kernel thông thường không chạy trực tiếp các SQL cell đặc thù SpurLab.
5. Output đi kèm là kết quả minh họa đã lưu cho ngày **07/10/2026**. Nó không tạo bảng tạm trong kernel mới. Hãy chạy lại bài học bằng connection của bạn để có dữ liệu và thời điểm tải mới.

Bản chia sẻ giữ hướng dẫn, 16 SQL cell và cấu hình biểu đồ; không mang theo cell bootstrap do máy tác giả sinh ra, session kernel hoặc thông tin đăng nhập.

## 0 · Làm quen với notebook và connection

| Thành phần | Dùng để làm gì? |
|---|---|
| Markdown cell | Viết câu hỏi, giải thích, nhận định và nguồn dữ liệu. |
| SQL cell | Chạy truy vấn DuckDB; kết quả hiện dưới cell. Chọn ngôn ngữ **SQL** khi thêm code cell. |
| Global connection `tcinvest` | Cấu hình kết nối dùng chung, có thể dùng lại ở nhiều notebook. |
| Connection trong notebook | Hiển thị nguồn đã gắn và các bảng/hàm của nó trong catalog. |
| Kernel / session | Giữ biến SQL và bảng tạm trong phiên làm việc. Khởi động lại thì cần chạy lại các cell tạo chúng. |

**Thao tác:** mở khu vực **Connections**, chọn connection có sẵn `tcinvest` và gắn vào notebook. Mở cây `tcinvest` để xem tên bảng/hàm, cột, tham số và câu SQL gợi ý. Bản trong repository chưa mang theo cấu hình connection của máy tác giả. Sau khi gắn nguồn, catalog cần có `ds://tcinvest`, gồm **55 bảng/hàm** tại thời điểm tạo bài học.

Khi bạn gắn connection, SpurLab tạo cell **SPUR datasource setup** do hệ thống quản lý. Nó chuẩn bị datasource và có thể hiển thị mã Python; người học chỉ viết **SQL** cho toàn bộ phân tích bên dưới. Nếu mở kernel mới, chạy cell setup trước.

**Cách học:** đọc Markdown → chạy SQL ngay bên dưới bằng nút chạy của cell → xem kết quả → thử thay một tham số. Lần đầu chạy từ trên xuống. Khi sửa tham số, chạy lại mọi cell phía sau; bảng tạm không tự cập nhật theo phần giải thích Markdown.

Luồng bài học: **khám phá nguồn → đặt tham số → tải và kiểm tra → tính chỉ báo → so sánh dòng tiền/ngành → viết nhận định**.

## 1 · SELECT đầu tiên: tìm mã ngành
**Câu hỏi:** connector cung cấp những ngành nào?

`tcinvest.get_industries` là view không cần tham số. `SELECT` chọn cột, `AS` đặt tên dễ đọc, `ORDER BY` sắp xếp. Mã ngành là chuỗi: giữ **'0500'** để không mất số 0 đầu.

**Kết quả cần thấy:** 19 dòng mã và tên ngành. Thử thêm `WHERE name ILIKE '%ngân%'` trước `ORDER BY` để chỉ lấy ngành ngân hàng, rồi bỏ bộ lọc trước khi tiếp tục.

In [4]:
-- SQL 01 | Khám phá view không tham số
SELECT code AS ma_nganh, name AS ten_nganh
FROM tcinvest.get_industries
ORDER BY name;

ma_nganh,ten_nganh
5300,Bán lẻ
8500,Bảo hiểm
8600,Bất động sản
9500,Công nghệ Thông tin
5700,Du lịch và Giải trí
0500,Dầu khí
8700,Dịch vụ tài chính
2700,Hàng & Dịch vụ Công nghiệp
3700,Hàng cá nhân & Gia dụng
1300,Hóa chất


## 2 · Chọn đúng bảng và cách gọi
**Câu hỏi:** muốn xem lịch sử giá, độ rộng hoặc dòng vốn thì dùng chức năng nào?

Truy vấn dưới đọc mô tả từ connector. Tên API trong kết quả và tên hàm SQL có thể khác cách viết; lấy **invoke SQL** từ catalog của từng bảng để biết cú pháp SQL chính xác.

| Bảng/hàm SQL dùng trong bài | Ý nghĩa |
|---|---|
| `tcinvest.get_industries` | Danh mục ngành, không cần đối số |
| `tcinvest_get_long_term_candle_chart(...)` | Lịch sử OHLCV, cần mã, loại, độ phân giải, mốc thời gian và số quan sát |
| `tcinvest_get_market_breadth(...)` | Số mã tăng/giảm/đứng giá |
| `tcinvest_get_market_foreign_val(...)` | Giá trị mua/bán ròng của khối ngoại |
| `tcinvest_get_flow_market_value_percent_trading(...)` | Chuỗi giá trị giao dịch `accValue` |
| `tcinvest_get_industry_index(...)` | Chuỗi chỉ số ngành |

**Phân biệt:** `FROM tcinvest.get_industries` đọc view; `FROM tcinvest_get_market_breadth(exchange='HOSE', industry='ALL')` gọi hàm trả bảng với đối số. Không đoán rằng mọi hàm đều có view không đối số.

In [5]:
-- SQL 02 | Tìm tài liệu chức năng từ chính nguồn
SELECT name, description
FROM tcinvest.list_tools
WHERE regexp_matches(
  name, 'LongTermCandleChart|MarketBreadth|MarketForeignVal|IndustryIndex|Industries|FlowMarketValuePercentTrading'
)
ORDER BY name;

name,description
tcinvest-getFlowMarketValuePercentTrading,"This API provides accumulated trading value relative to market capitalization. Used for evaluating liquidity and money flow intensity by industry.\n\nPARAMETER CONSTRAINT:\n- If exchange is a market index (HOSE, HNX, UPCOM, VN30, HNX30), then industry MUST be ""ALL"".\n- If industry is an ICB L2 code (e.g. 8600, 0500, 8300), then exchange MUST be ""ALL"".\n- exchange and industry cannot both be specific values at the same time; one of them must always be ""ALL"".\n\nKey response fields:\n- body[].seqTime: Timestamp in dd/MM/yyyy format\n- body[].accValue: Accumulated trading value in VND\n- body[].marketCap: Industry market capitalization in VND\n- body[].marketCapPercent: Industry market cap weight vs total market (0-1, e.g. 0.31 = 31%)\n- body[].accValuePercent: Accumulated trading value / Market cap ratio (0-1)\n"
tcinvest-getIndustries,"Returns the list of all industry classifications available in the Vietnamese stock market.\nSupports Vietnamese and English language for industry names.\n\nResponse fields:\n- listIndustry: Array containing industry items\n- id: Unique industry identifier (integer)\n- name: Industry name (in Vietnamese or English depending on language header)\n- code: ICB industry code level 2 (4-digit string, e.g. ""8300"" for Banking). This value is used as the icbCodeL2 path parameter in industry-level financial APIs (e.g. /finance/industry/{icbCodeL2}/incomestatement, /finance/industry/{icbCodeL2}/financialratio, etc.)\n"
tcinvest-getIndustryIndex,"This API provides industry index time series data showing index value and trading volume at each time point. Used for charting industry index movements (prism), flows throughout the trading day.\n\nIMPORTANT - Usage guidance:\n- When user asks about ""chỉ số ngành"" (industry index/indicator), ""biến động ngành"" (industry movement), or ""giá ngành"" (industry price), use ONLY this API (getIndustryIndex).\n- Do NOT call getIncomeStatementIndustryForBank, getFinancialRatioIndustryForBank, or getBalanceSheetIndustryForBank unless user explicitly asks for financial metrics like ROE, NIM, profit, revenue, or balance sheet data.\n- For market breadth/sentiment of an industry, use getMarketBreadth.\n- For top contributing stocks in an industry, use getMarketLeader.\n- For foreign investor flow in an industry, use getMarketForeignVal.\n- For money flow intensity, use getFlowMarketValuePercentTrading.\n- For cash flow index comparison across industries, use getFiTimeSeries.\n- For heat-map visualization data, use getMarketVolatility.\n\nPARAMETER CONSTRAINT:\n- If exchange is a market index (HOSE, HNX, UPCOM, VN30, HNX30), then industry MUST be ""ALL"".\n- If industry is an ICB L2 code (e.g. 8600, 0500, 8300), then exchange MUST be ""ALL"".\n- exchange and industry cannot both be specific values at the same time; one of them must always be ""ALL"".\n\nKey response fields:\n- exchange: Exchange code queried\n- industry: Industry code queried\n- referencePrice: Reference price level (draws horizontal reference line on chart)\n- data[].i: Index value at time point (points)\n- data[].v: Trading volume at time point (shares)\n- data[].s: Timestamp in dd/MM/yyyy format\n"
tcinvest-getLongTermCandleChart,"Returns historical candlestick price bars for a given ticker with Daily, Weekly, or Monthly resolution. Use this to analyze long-term price trends, draw charts, or compute technical indicators over extended periods.\n\nParameters guide:\n- type: Use ""stock"" for listed equities (e.g. TCB, VCB, FPT), ""index"" for market indices (e.g. VNINDEX, HNX, VN30)\n- ticker: The symbol code, max 10 characters\n- resolution: ""D"" for daily bars, ""W"" for weekly bars, ""M"" for monthly bars\n- to: Unix timestamp in seconds of the START date (the most recent date you want data from, inclusive).\n Example: today is 05-05-2026, Unix timestamp = 1777939200\n- countBack: Number of bars to return going BACKWARDS from the `to` dat

## 3 · Một cell tham số cho cả bài
**Câu hỏi:** làm thế nào đổi kỳ phân tích mà không sửa ngày ở nhiều chỗ?

`SET VARIABLE` lưu biến trong session; `getvariable(...)` đọc lại biến. Ngày chốt mặc định là **2026-10-07**; `tut_ticker` nhận mã chỉ số, ví dụ **VNINDEX**, **VN30**, **HNXINDEX**, **UPCOM**. Các phần HOSE và ngành là bối cảnh thị trường riêng; đổi mã chỉ số không tự đổi sàn trong những phần đó.

`epoch` chuyển mốc ngày sang số giây Unix để truyền cho tham số `"to"`. Dấu nháy kép dùng cho tên `"to"`; dấu nháy đơn dùng cho giá trị chuỗi. Các tham số API của hàm lịch sử được khai báo dạng VARCHAR, nên số phiên được truyền thành chuỗi.

**Thực hành:** giữ nguyên cấu hình lần chạy đầu. Muốn cập nhật báo cáo, chọn một ngày đã kết thúc giao dịch và chạy lại từ cell này đến cuối. [Tài liệu biến SQL của DuckDB](https://www.duckdb.org/docs/current/sql/statements/set_variable).

In [6]:
-- SQL 03 | Chỉ sửa cấu hình tại đây
SET VARIABLE tut_as_of = DATE '2026-10-07';
SET VARIABLE tut_ticker = 'VNINDEX';
SET VARIABLE tut_countback = '250';

SELECT version() AS duckdb_version,
       getvariable('tut_as_of') AS ngay_chot,
       getvariable('tut_ticker') AS chi_so,
       getvariable('tut_countback') AS so_quan_sat_yeu_cau,
       CAST(epoch(getvariable('tut_as_of')) AS BIGINT)::VARCHAR AS to_epoch;

duckdb_version,ngay_chot,chi_so,so_quan_sat_yeu_cau,to_epoch
v1.5.5,2026-10-07,VNINDEX,250,1791331200


## 4 · Gọi connector và lưu một bản dữ liệu trong session
**Câu hỏi:** làm sao thử nhiều công thức mà không gọi lại API mỗi lần?

`CREATE OR REPLACE TEMP TABLE` lưu kết quả vào bảng tạm. Cell này gọi TCInvest và lưu `tut_prices_raw`; các cell sau đọc lại bảng đó. Chạy lại cell này sẽ thay bản dữ liệu tạm bằng kết quả mới.

Các trường OHLC là giá mở cửa/cao nhất/thấp nhất/đóng cửa; ở đây `type='index'` nên giá trị là **điểm chỉ số**. `volume` là khối lượng, không phải giá trị tiền. `tradingdate` ban đầu là chuỗi.

**Kết quả cần thấy:** 5 quan sát mới nhất của mẫu tải về. `LIMIT 5` chỉ giới hạn bảng xem trước ở câu SELECT cuối; nó không giảm 250 quan sát đã tải vào bảng tạm. `loaded_at` ghi thời điểm truy vấn, khác ngày giao dịch.

Bảng tạm mất khi kernel khởi động lại. Output đã lưu trong notebook vẫn có thể nhìn thấy, nhưng không chứng minh bảng tạm còn tồn tại.

In [7]:
-- SQL 04 | Tải nguồn một lần, tái sử dụng ở nhiều cell
CREATE OR REPLACE TEMP TABLE tut_prices_raw AS
SELECT *,
       getvariable('tut_as_of') AS requested_as_of,
       getvariable('tut_ticker') AS requested_ticker,
       current_timestamp AS loaded_at
FROM tcinvest_get_long_term_candle_chart(
  ticker=getvariable('tut_ticker'),
  type='index',
  resolution='D',
  "to"=CAST(epoch(getvariable('tut_as_of')) AS BIGINT)::VARCHAR,
  countback=getvariable('tut_countback')
);

SELECT tradingdate, open, high, low, close, volume, loaded_at
FROM tut_prices_raw
ORDER BY TRY_CAST(tradingdate AS TIMESTAMP) DESC NULLS LAST
LIMIT 5;

tradingdate,open,high,low,close,volume,loaded_at
2026-10-07T00:00:00.000Z,1760.54,1760.54,1743.24,1753.39,486855603,2026-10-08 09:26:26.544501+07:00
2026-10-06T00:00:00.000Z,1755.34,1763.84,1734.94,1759.08,648123872,2026-10-08 09:26:26.544501+07:00
2026-10-05T00:00:00.000Z,1738.92,1756.30,1738.78,1753.20,433782122,2026-10-08 09:26:26.544501+07:00
2026-10-02T00:00:00.000Z,1747.29,1751.60,1731.03,1737.71,616671876,2026-10-08 09:26:26.544501+07:00
2026-10-01T00:00:00.000Z,1765.71,1768.05,1738.39,1749.30,450474491,2026-10-08 09:26:26.544501+07:00


## 5 · Kiểm tra dữ liệu trước khi tính toán
**Câu hỏi:** mẫu có đủ dài, đúng kỳ và có trùng ngày không?

`TRY_CAST` trả NULL khi không đổi được kiểu, giúp đếm lỗi thay vì dừng ngay. Không sắp xếp ngày dạng chuỗi `dd/mm/yy` theo chữ cái. Kiểm tra số dòng, số ngày khác nhau, NULL, giá/khối lượng không hợp lệ và quan hệ OHLC.

**Kỳ vọng với VNINDEX:** 250 dòng, ngày cuối 07/10/2026, không trùng ngày và không lỗi OHLC. Nếu đổi sang UPCOM, mẫu trước đó có 2 bất nhất OHLC; phải xem lại nguồn trước khi dùng mô hình dựa vào high/low.

Ngày cuối nhỏ hơn ngày chốt có thể do ngày nghỉ hoặc nguồn chậm; cần kiểm tra lịch giao dịch trước khi kết luận. `countback=250` là số yêu cầu, không phải cam kết nguồn luôn trả đủ.

In [8]:
-- SQL 05 | Audit toàn bộ dữ liệu thô
WITH parsed AS (
  SELECT *, TRY_CAST(tradingdate AS TIMESTAMP)::DATE AS ngay
  FROM tut_prices_raw
)
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT ngay) AS so_ngay_khac_nhau,
       MIN(ngay) AS ngay_dau, MAX(ngay) AS ngay_cuoi,
       COUNT(*) FILTER (WHERE ngay IS NULL) AS loi_ngay,
       COUNT(ngay)-COUNT(DISTINCT ngay) AS ngay_trung,
       COUNT(*) FILTER (WHERE close IS NULL OR close <= 0
                              OR volume IS NULL OR volume < 0) AS loi_gia_khoi_luong,
       COUNT(*) FILTER (WHERE open IS NULL OR high IS NULL OR low IS NULL
          OR low > LEAST(open, close, high)
          OR high < GREATEST(open, close, low)) AS ohlc_bat_thuong,
       COUNT(*) FILTER (WHERE ngay > getvariable('tut_as_of')) AS dong_sau_ngay_chot,
       MIN(requested_as_of) AS ngay_chot_luc_tai,
       MIN(requested_ticker) AS ma_luc_tai
FROM parsed;

so_dong,so_ngay_khac_nhau,ngay_dau,ngay_cuoi,loi_ngay,ngay_trung,loi_gia_khoi_luong,ohlc_bat_thuong,dong_sau_ngay_chot,ngay_chot_luc_tai,ma_luc_tai
250,250,2025-10-06,2026-10-07,0,0,0,0,0,2026-10-07,VNINDEX


## 6 · Window functions: lợi suất và trung bình động
**Câu hỏi:** chỉ số đang tăng hay giảm so với các phiên trước?

Cell này tạo hai bảng tạm: `tut_prices` chuẩn hóa ngày và `tut_metrics` chứa chỉ báo. Nó dừng với thông báo nếu cấu hình tải không khớp, có ngày trùng hoặc dữ liệu đóng cửa/khối lượng không hợp lệ; không tự điền giá thiếu.

- `LAG(close, 5)`: giá cách 5 **quan sát giao dịch**, không phải 5 ngày lịch; cần chuỗi phiên không bị thiếu.
- Lợi suất: **100 × (giá hiện tại / giá trước đó − 1)**. Muốn lợi suất 20 phiên cần ít nhất **21 giá**.
- MA20: trung bình của phiên hiện tại và 19 phiên trước. Chỉ xuất MA20 khi có đủ 20 quan sát.
- Trung bình khối lượng so sánh: **20 phiên trước, loại phiên hiện tại**.
- `NULLIF(mau_so, 0)` ngăn chia cho 0. NULL nghĩa là chưa đủ cơ sở tính, không phải lợi suất bằng 0.

Tính cửa sổ trên toàn bộ lịch sử rồi mới lấy các dòng cuối. Nếu đặt `WHERE ngay = ngay_chot` trước `LAG`, bạn sẽ xóa lịch sử cần dùng. [Tài liệu window functions](https://www.duckdb.org/docs/current/sql/functions/window_functions).

In [9]:
-- SQL 06 | Chuẩn hóa và tạo chỉ báo; chỉ dùng dữ liệu đến ngày chốt
SELECT CASE
  WHEN COUNT(*) = 0 THEN error('Không có dữ liệu; chạy lại SQL 04.')
  WHEN MIN(requested_as_of) <> getvariable('tut_as_of')
    OR MIN(requested_ticker) <> getvariable('tut_ticker')
    THEN error('Cấu hình đã đổi; hãy chạy lại SQL 04.')
  WHEN COUNT(*) FILTER (
    WHERE TRY_CAST(tradingdate AS TIMESTAMP) IS NULL
       OR close IS NULL OR close <= 0 OR volume IS NULL OR volume < 0
  ) > 0 THEN error('Có dữ liệu không hợp lệ; xem SQL 05.')
  WHEN COUNT(*) <> COUNT(DISTINCT TRY_CAST(tradingdate AS TIMESTAMP)::DATE)
    THEN error('Có ngày trùng; kiểm tra nguồn trước khi tính LAG.')
  ELSE 'OK' END AS kiem_tra
FROM tut_prices_raw;

CREATE OR REPLACE TEMP TABLE tut_prices AS
SELECT requested_ticker AS chi_so,
       TRY_CAST(tradingdate AS TIMESTAMP)::DATE AS ngay,
       open, high, low, close, volume
FROM tut_prices_raw
WHERE TRY_CAST(tradingdate AS TIMESTAMP)::DATE <= getvariable('tut_as_of');

CREATE OR REPLACE TEMP TABLE tut_metrics AS
SELECT *,
       LAG(close, 1) OVER w AS gia_truoc_1,
       LAG(close, 5) OVER w AS gia_truoc_5,
       LAG(close, 20) OVER w AS gia_truoc_20,
       CASE WHEN COUNT(*) OVER w20 = 20
            THEN AVG(close) OVER w20 END AS ma20,
       CASE WHEN COUNT(*) OVER w50 = 50
            THEN AVG(close) OVER w50 END AS ma50,
       CASE WHEN COUNT(*) OVER v20 = 20
            THEN AVG(volume) OVER v20 END AS kl_tb20_truoc
FROM tut_prices
WINDOW w AS (ORDER BY ngay),
       w20 AS (ORDER BY ngay ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
       w50 AS (ORDER BY ngay ROWS BETWEEN 49 PRECEDING AND CURRENT ROW),
       v20 AS (ORDER BY ngay ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING);

SELECT ngay, close, gia_truoc_1,
       ROUND(100*(close/NULLIF(gia_truoc_1,0)-1),2) AS loi_suat_1_phien_pct,
       ROUND(ma20,2) AS ma20, ROUND(ma50,2) AS ma50
FROM tut_metrics
ORDER BY ngay DESC
LIMIT 10;

ngay,close,gia_truoc_1,loi_suat_1_phien_pct,ma20,ma50
2026-10-07,1753.39,1759.08,-0.32,1786.53,1777.33
2026-10-06,1759.08,1753.20,0.34,1790.21,1775.98
2026-10-05,1753.20,1737.71,0.89,1793.78,1774.79
2026-10-02,1737.71,1749.30,-0.66,1797.20,1773.10
2026-10-01,1749.30,1768.62,-1.09,1802.97,1772.95
2026-09-30,1768.62,1777.73,-0.51,1806.89,1772.84
2026-09-29,1777.73,1780.68,-0.17,1810.07,1773.21
2026-09-28,1780.68,1785.11,-0.25,1812.76,1773.74
2026-09-25,1785.11,1775.09,0.56,1814.79,1773.77
2026-09-24,1775.09,1801.65,-1.47,1815.11,1774.20


## 7 · Biến bảng số thành một nhận định
**Câu hỏi:** tại ngày mới nhất trong mẫu, giá đứng ở đâu so với MA20/MA50?

Cell dưới giữ một dòng mới nhất **sau khi** tính chỉ báo. `CASE` chuyển điều kiện số thành một nhãn mô tả để hỗ trợ đọc kết quả.

**Mốc tự đối chiếu với cấu hình mặc định:** VN-Index 1.753,39; lợi suất 20 phiên −4,04%; MA20 1.786,53; MA50 1.777,33. Giá dưới cả hai MA cho thấy động lượng yếu theo thước đo này. Nó không tự chứng minh thị trường rẻ/đắt hoặc dự báo phiên kế tiếp.

Thử đổi sang VN30 tại SQL 03 và chạy lại SQL 04–07. Ghi nhận điều gì thay đổi; giữ nguyên công thức để so sánh công bằng.

In [10]:
-- SQL 07 | Một dòng để viết nhận định
SELECT chi_so, ngay,
       ROUND(close,2) AS dong_cua,
       ROUND(100*(close/NULLIF(gia_truoc_1,0)-1),2) AS loi_suat_1_phien_pct,
       ROUND(100*(close/NULLIF(gia_truoc_5,0)-1),2) AS loi_suat_5_phien_pct,
       ROUND(100*(close/NULLIF(gia_truoc_20,0)-1),2) AS loi_suat_20_phien_pct,
       ROUND(ma20,2) AS ma20, ROUND(ma50,2) AS ma50,
       ROUND(volume/1e6,2) AS khoi_luong_trieu,
       ROUND(volume/NULLIF(kl_tb20_truoc,0),2) AS kl_so_voi_tb20_lan,
       CASE WHEN ma20 IS NULL OR ma50 IS NULL THEN 'Chưa đủ cửa sổ MA'
            WHEN close < ma20 AND close < ma50 THEN 'Dưới cả MA20 và MA50'
            WHEN close > ma20 AND close > ma50 THEN 'Trên cả MA20 và MA50'
            ELSE 'Tín hiệu MA chưa đồng thuận' END AS mo_ta_xu_huong,
       CASE WHEN ngay = getvariable('tut_as_of') THEN 'Khớp ngày chốt'
            ELSE 'Kiểm tra ngày nghỉ hoặc độ trễ nguồn' END AS kiem_dinh_ngay
FROM tut_metrics
ORDER BY ngay DESC
LIMIT 1;

chi_so,ngay,dong_cua,loi_suat_1_phien_pct,loi_suat_5_phien_pct,loi_suat_20_phien_pct,ma20,ma50,khoi_luong_trieu,kl_so_voi_tb20_lan,mo_ta_xu_huong,kiem_dinh_ngay
VNINDEX,2026-10-07,1753.39,-0.32,-0.86,-4.04,1786.53,1777.33,486.86,0.93,Dưới cả MA20 và MA50,Khớp ngày chốt


## 8 · Thanh khoản: so sánh với chính lịch sử của sàn
**Câu hỏi:** giá trị giao dịch HOSE cao hay thấp so với bình thường?

Nguồn này dùng `seqtime` dạng ngày/tháng/năm. `COALESCE` thử định dạng ngày có hoặc không có giờ, với năm 2 hoặc 4 chữ số; `accvalue / 1e9` đổi đồng sang **tỷ đồng**. Cửa sổ bình quân loại phiên hiện tại để tránh tự pha loãng mức so sánh.

`tut_liquidity_raw` giữ nguyên nguồn; `tut_liquidity` lưu kết quả theo ngày. Số dòng lỗi/trùng vẫn được hiển thị. Nếu có lỗi, không dùng tỷ lệ cửa sổ trước khi xử lý.

**Mốc mẫu:** 12.483,27 tỷ đồng, khoảng **0,95 lần** trung bình 20 phiên trước. `accValue` là phạm vi trường nguồn, chưa xác nhận có bao gồm toàn bộ giao dịch thỏa thuận; không gọi đây là tổng thanh khoản tất cả sàn.

In [18]:
-- SQL 08 | Giá trị giao dịch HOSE
CREATE OR REPLACE TEMP TABLE tut_liquidity_raw AS
SELECT * FROM tcinvest_get_flow_market_value_percent_trading(
  exchange='HOSE', industry='ALL'
);

CREATE OR REPLACE TEMP TABLE tut_liquidity AS
WITH parsed AS (
 SELECT COALESCE(TRY_STRPTIME(seqtime,'%d/%m/%y'),
                 TRY_STRPTIME(seqtime,'%d/%m/%Y'),
                  TRY_STRPTIME(seqtime,'%d/%m/%y %H:%M'),
                  TRY_STRPTIME(seqtime,'%d/%m/%Y %H:%M'))::DATE AS ngay,
        accvalue::DOUBLE AS gia_tri
 FROM tut_liquidity_raw
)
SELECT *,
       LAG(gia_tri) OVER (ORDER BY ngay) AS gia_tri_truoc,
       COUNT(gia_tri) OVER w AS n20,
       AVG(gia_tri) OVER w AS tb20_truoc
FROM parsed
WHERE ngay <= getvariable('tut_as_of')
WINDOW w AS (ORDER BY ngay ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING);

WITH audit AS (
 SELECT COUNT(*) FILTER (WHERE ngay IS NULL OR gia_tri IS NULL OR gia_tri < 0) AS loi_du_lieu,
        COUNT(ngay)-COUNT(DISTINCT ngay) AS ngay_trung
 FROM (
  SELECT COALESCE(TRY_STRPTIME(seqtime,'%d/%m/%y'),
                  TRY_STRPTIME(seqtime,'%d/%m/%Y'),
                  TRY_STRPTIME(seqtime,'%d/%m/%y %H:%M'),
                  TRY_STRPTIME(seqtime,'%d/%m/%Y %H:%M'))::DATE AS ngay,
         accvalue AS gia_tri
  FROM tut_liquidity_raw
 )
)
SELECT ngay, ROUND(gia_tri/1e9,2) AS gia_tri_ty_dong,
       CASE WHEN n20=20 THEN ROUND(tb20_truoc/1e9,2) END AS tb20_truoc_ty_dong,
       CASE WHEN n20=20 AND loi_du_lieu=0 AND ngay_trung=0
            THEN ROUND(gia_tri/NULLIF(tb20_truoc,0),2) END AS so_voi_tb20_lan,
       ROUND(100*(gia_tri/NULLIF(gia_tri_truoc,0)-1),2) AS thay_doi_1_phien_pct,
       n20 AS so_phien_cua_so, loi_du_lieu, ngay_trung
FROM tut_liquidity CROSS JOIN audit
ORDER BY ngay DESC LIMIT 1;

ngay,gia_tri_ty_dong,tb20_truoc_ty_dong,so_voi_tb20_lan,thay_doi_1_phien_pct,so_phien_cua_so,loi_du_lieu,ngay_trung
2026-10-07,12483.27,13143.74,0.95,-13.84,20,0,0


## 9 · Độ rộng thị trường: UNION ALL và tỷ lệ
**Câu hỏi:** chỉ số giảm có đồng nghĩa phần lớn cổ phiếu đều giảm không?

`UNION ALL` ghép ba tập có cùng cột; thêm nhãn sàn trước khi ghép. Theo schema connector: `a` = số mã tăng, `d` = giảm, `s` = đứng giá, `t` = ngày.

`ROW_NUMBER() OVER (PARTITION BY san ORDER BY ngay DESC)` đánh số từ ngày mới nhất cho từng sàn. `QUALIFY ... = 1` giữ dòng cuối sau khi tính cửa sổ. Bảng `expected` và `LEFT JOIN` giữ cả sàn bị thiếu dữ liệu để không bỏ sót nó.

Tỷ lệ tăng ở bài này = **tăng / (tăng + giảm + đứng giá)**. Một nơi khác dùng tăng/(tăng+giảm) sẽ ra số khác: luôn công bố mẫu số.

**Mốc HOSE:** 150 tăng, 151 giảm, 64 đứng giá; 41,10% số mã quan sát tăng. Đó là độ rộng khá cân bằng, dù chỉ số giảm.

In [19]:
-- SQL 09 | Ghép ba sàn và giữ ngày mới nhất trong kỳ
CREATE OR REPLACE TEMP TABLE tut_breadth_raw AS
SELECT 'HOSE' AS san, * FROM tcinvest_get_market_breadth(exchange='HOSE',industry='ALL')
UNION ALL
SELECT 'HNX' AS san, * FROM tcinvest_get_market_breadth(exchange='HNX',industry='ALL')
UNION ALL
SELECT 'UPCOM' AS san, * FROM tcinvest_get_market_breadth(exchange='UPCOM',industry='ALL');

CREATE OR REPLACE TEMP TABLE tut_breadth AS
WITH parsed AS (
 SELECT san, COALESCE(TRY_STRPTIME(t,'%d/%m/%y'),
                      TRY_STRPTIME(t,'%d/%m/%Y'),
                  TRY_STRPTIME(t,'%d/%m/%y %H:%M'),
                  TRY_STRPTIME(t,'%d/%m/%Y %H:%M'))::DATE AS ngay,
        a AS tang, d AS giam, s AS dung_gia
 FROM tut_breadth_raw
),
audit AS (
 SELECT san,
        COUNT(*) FILTER (WHERE ngay IS NULL OR tang IS NULL OR giam IS NULL
                         OR dung_gia IS NULL OR LEAST(tang,giam,dung_gia)<0) AS loi_du_lieu,
        COUNT(ngay)-COUNT(DISTINCT ngay) AS ngay_trung
 FROM parsed GROUP BY san
),
latest AS (
 SELECT * FROM parsed WHERE ngay <= getvariable('tut_as_of')
 QUALIFY ROW_NUMBER() OVER (PARTITION BY san ORDER BY ngay DESC)=1
),
expected(san) AS (VALUES ('HOSE'),('HNX'),('UPCOM'))
SELECT e.san, l.ngay, tang, giam, dung_gia,
       tang+giam+dung_gia AS so_ma_quan_sat,
       ROUND(100.0*tang/NULLIF(tang+giam+dung_gia,0),2) AS ty_le_tang_pct,
       ROUND(1.0*tang/NULLIF(giam,0),2) AS ty_so_tang_giam,
       a.loi_du_lieu, a.ngay_trung,
       CASE WHEN l.ngay IS NULL THEN 'Thiếu dữ liệu'
            WHEN a.loi_du_lieu>0 OR a.ngay_trung>0 THEN 'Kiểm tra dữ liệu gốc'
            WHEN l.ngay<>getvariable('tut_as_of') THEN 'Kiểm tra ngày nguồn'
            ELSE 'Đúng kỳ' END AS kiem_dinh
FROM expected e LEFT JOIN latest l USING(san) LEFT JOIN audit a USING(san);

SELECT * FROM tut_breadth ORDER BY san;

san,ngay,tang,giam,dung_gia,so_ma_quan_sat,ty_le_tang_pct,ty_so_tang_giam,loi_du_lieu,ngay_trung,kiem_dinh
HNX,2026-10-07,81,66,49,196,41.33,1.23,0,0,Đúng kỳ
HOSE,2026-10-07,150,151,64,365,41.10,0.99,0,0,Đúng kỳ
UPCOM,2026-10-07,100,85,92,277,36.10,1.18,0,0,Đúng kỳ


## 10 · Khối ngoại: tổng trượt 5 và 20 phiên
**Câu hỏi:** dòng vốn ngoại là biến động một ngày hay kéo dài nhiều phiên?

Trong nguồn này, `v` là giá trị ròng bằng đồng; dương = mua ròng, âm = bán ròng. Chia **1e9** để đọc theo tỷ đồng. Không cộng phần trăm để thay cho giá trị ròng.

`ROWS BETWEEN 4 PRECEDING AND CURRENT ROW` gồm **5 dòng**; tương tự 19 + hiện tại = **20 dòng**. Phải sắp xếp theo ngày đã parse vì thứ tự API có thể không theo thời gian.

Cell chỉ xuất tổng 5/20 phiên nếu đủ số quan sát và không phát hiện lỗi/trùng ngày. Các cửa sổ là số quan sát giao dịch trong nguồn; kiểm tra thêm lịch sàn nếu nghi ngờ thiếu phiên.

**Mốc HOSE:** khoảng −877,65 tỷ trong phiên; −7.196,70 tỷ trong 5 phiên; −9.201,27 tỷ trong 20 phiên. Các sàn có quy mô khác nhau; không so sức ép chỉ bằng trị tuyệt đối. Khi tổng hợp, kiểm tra chúng cùng ngày trước.

In [20]:
-- SQL 10 | Khối ngoại, giá trị tính bằng tỷ đồng
CREATE OR REPLACE TEMP TABLE tut_foreign_raw AS
SELECT 'HOSE' AS san, * FROM tcinvest_get_market_foreign_val(exchange='HOSE',industry='ALL')
UNION ALL
SELECT 'HNX' AS san, * FROM tcinvest_get_market_foreign_val(exchange='HNX',industry='ALL')
UNION ALL
SELECT 'UPCOM' AS san, * FROM tcinvest_get_market_foreign_val(exchange='UPCOM',industry='ALL');

CREATE OR REPLACE TEMP TABLE tut_foreign AS
WITH parsed AS (
 SELECT san, COALESCE(TRY_STRPTIME(t,'%d/%m/%y'),
                      TRY_STRPTIME(t,'%d/%m/%Y'),
                  TRY_STRPTIME(t,'%d/%m/%y %H:%M'),
                  TRY_STRPTIME(t,'%d/%m/%Y %H:%M'))::DATE AS ngay,
        v::DOUBLE/1e9 AS rong_ty
 FROM tut_foreign_raw
),
audit AS (
 SELECT san, COUNT(*) FILTER (WHERE ngay IS NULL OR rong_ty IS NULL) AS loi_du_lieu,
        COUNT(ngay)-COUNT(DISTINCT ngay) AS ngay_trung
 FROM parsed GROUP BY san
),
metrics AS (
 SELECT *, SUM(rong_ty) OVER w5 AS rong_5,
           SUM(rong_ty) OVER w20 AS rong_20,
           COUNT(rong_ty) OVER w5 AS n5,
           COUNT(rong_ty) OVER w20 AS n20,
           COUNT(*) FILTER (WHERE rong_ty<0) OVER w20 AS phien_ban_20
 FROM parsed WHERE ngay<=getvariable('tut_as_of')
 WINDOW w5 AS (PARTITION BY san ORDER BY ngay ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        w20 AS (PARTITION BY san ORDER BY ngay ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
),
latest AS (
 SELECT * FROM metrics
 QUALIFY ROW_NUMBER() OVER (PARTITION BY san ORDER BY ngay DESC)=1
),
expected(san) AS (VALUES ('HOSE'),('HNX'),('UPCOM'))
SELECT e.san, l.ngay, ROUND(rong_ty,2) AS rong_1_phien_ty,
       CASE WHEN n5=5 AND loi_du_lieu=0 AND ngay_trung=0
            THEN ROUND(rong_5,2) END AS rong_5_phien_ty,
       CASE WHEN n20=20 AND loi_du_lieu=0 AND ngay_trung=0
            THEN ROUND(rong_20,2) END AS rong_20_phien_ty,
       n20, phien_ban_20, loi_du_lieu, ngay_trung,
       CASE WHEN l.ngay IS NULL THEN 'Thiếu dữ liệu'
            WHEN loi_du_lieu>0 OR ngay_trung>0 THEN 'Kiểm tra dữ liệu gốc'
            WHEN l.ngay<>getvariable('tut_as_of') THEN 'Kiểm tra ngày nguồn'
            WHEN n20<20 THEN 'Thiếu cửa sổ 20 phiên'
            ELSE 'Đúng kỳ' END AS kiem_dinh
FROM expected e LEFT JOIN latest l USING(san) LEFT JOIN audit a USING(san);

SELECT * FROM tut_foreign ORDER BY san;

san,ngay,rong_1_phien_ty,rong_5_phien_ty,rong_20_phien_ty,n20,phien_ban_20,loi_du_lieu,ngay_trung,kiem_dinh
HNX,2026-10-07,-30.89,5.09,-58.64,20,13,0,0,Đúng kỳ
HOSE,2026-10-07,-877.65,-7196.70,-9201.27,20,13,0,0,Đúng kỳ
UPCOM,2026-10-07,9.37,3023.58,3226.64,20,4,0,0,Đúng kỳ


## 11 · Mở rộng ra 19 ngành: tải nguồn và kiểm tra độ phủ
**Câu hỏi:** ngành nào mạnh hơn, và có ngành nào bị thiếu trong phép so sánh?

Đây là phần nâng cao, có nhiều lời gọi nguồn hơn các bài trước. Mỗi nhánh `UNION ALL` gọi một mã ngành từ danh mục của connector. Với mã ngành ICB cụ thể, bài này dùng `exchange='ALL'`. Danh sách mã được viết tường minh để mỗi lời gọi có đối số cố định.

Giữ danh mục `tut_industry_catalog` riêng, rồi `LEFT JOIN` với kết quả tải để ngành thiếu vẫn xuất hiện. Nếu nguồn thêm mã mới ngoài 19 mã này, bảng kiểm tra sẽ hiện số quan sát bằng 0 và bạn cần bổ sung lời gọi tương ứng.

**Schema quan trọng:** `s` = ngày, `i` = điểm chỉ số ngành. Không so điểm số tuyệt đối giữa hai ngành; mức gốc của mỗi chuỗi khác nhau.

**Kết quả cần thấy ở kỳ mẫu:** 19 ngành, mỗi ngành 248 quan sát đến 07/10/2026; không lỗi ngày hay ngày trùng. Sau khi tải xong, phần xếp hạng và biểu đồ chỉ dùng bảng tạm, không gọi lại 19 API.

In [21]:
-- SQL 11 | Tải 19 ngành một lần
CREATE OR REPLACE TEMP TABLE tut_industry_catalog AS
SELECT code AS ma_nganh, name AS ten_nganh FROM tcinvest.get_industries;

CREATE OR REPLACE TEMP TABLE tut_industry_raw AS
SELECT '0500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='0500')
UNION ALL
SELECT '1300' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='1300')
UNION ALL
SELECT '1700' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='1700')
UNION ALL
SELECT '2300' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='2300')
UNION ALL
SELECT '2700' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='2700')
UNION ALL
SELECT '3300' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='3300')
UNION ALL
SELECT '3500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='3500')
UNION ALL
SELECT '3700' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='3700')
UNION ALL
SELECT '4500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='4500')
UNION ALL
SELECT '5300' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='5300')
UNION ALL
SELECT '5500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='5500')
UNION ALL
SELECT '5700' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='5700')
UNION ALL
SELECT '6500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='6500')
UNION ALL
SELECT '7500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='7500')
UNION ALL
SELECT '8300' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='8300')
UNION ALL
SELECT '8500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='8500')
UNION ALL
SELECT '8600' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='8600')
UNION ALL
SELECT '8700' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='8700')
UNION ALL
SELECT '9500' AS ma_nganh, * FROM tcinvest_get_industry_index(exchange='ALL',industry='9500');

CREATE OR REPLACE TEMP TABLE tut_industry_prices AS
SELECT ma_nganh,
       COALESCE(TRY_STRPTIME(s,'%d/%m/%y'),
                TRY_STRPTIME(s,'%d/%m/%Y'),
                  TRY_STRPTIME(s,'%d/%m/%y %H:%M'),
                  TRY_STRPTIME(s,'%d/%m/%Y %H:%M'))::DATE AS ngay,
       i::DOUBLE AS chi_so_nganh
FROM tut_industry_raw;

SELECT c.ma_nganh, c.ten_nganh,
       COUNT(p.ma_nganh) AS so_dong_nguon,
       COUNT(*) FILTER (WHERE p.ngay<=getvariable('tut_as_of')) AS so_phien_den_ngay_chot,
       COUNT(*) FILTER (WHERE p.ngay>getvariable('tut_as_of')) AS dong_sau_ngay_chot,
       MAX(p.ngay) FILTER (WHERE p.ngay<=getvariable('tut_as_of')) AS ngay_cuoi_trong_ky,
       COUNT(*) FILTER (WHERE p.ma_nganh IS NOT NULL AND
                  (p.ngay IS NULL OR p.chi_so_nganh IS NULL OR p.chi_so_nganh<=0)) AS loi_du_lieu,
       COUNT(p.ngay)-COUNT(DISTINCT p.ngay) AS ngay_trung
FROM tut_industry_catalog c LEFT JOIN tut_industry_prices p USING(ma_nganh)
GROUP BY c.ma_nganh,c.ten_nganh
ORDER BY c.ten_nganh;

ma_nganh,ten_nganh,so_dong_nguon,so_phien_den_ngay_chot,dong_sau_ngay_chot,ngay_cuoi_trong_ky,loi_du_lieu,ngay_trung
5300,Bán lẻ,249,248,1,2026-10-07,0,0
8500,Bảo hiểm,249,248,1,2026-10-07,0,0
8600,Bất động sản,249,248,1,2026-10-07,0,0
9500,Công nghệ Thông tin,249,248,1,2026-10-07,0,0
5700,Du lịch và Giải trí,249,248,1,2026-10-07,0,0
0500,Dầu khí,249,248,1,2026-10-07,0,0
8700,Dịch vụ tài chính,249,248,1,2026-10-07,0,0
2700,Hàng & Dịch vụ Công nghiệp,249,248,1,2026-10-07,0,0
3700,Hàng cá nhân & Gia dụng,249,248,1,2026-10-07,0,0
1300,Hóa chất,249,248,1,2026-10-07,0,0


### 11b · Tình huống thực tế: cùng cột có cả ngày và ngày–giờ

Khi tạo tutorial, connector ngành trả 248 dòng lịch sử và 1 dòng trong phiên, ví dụ **`08/10/26 09:30`**. Chỉ parse định dạng `%d/%m/%y` sẽ biến dòng này thành NULL và có thể khiến bảng xếp hạng bị loại hết.

Truy vấn SQL 11 đã thử cả định dạng có giờ **`%d/%m/%y %H:%M`**, rồi mới chuyển về DATE. Bảng kiểm tra giữ cột `dong_sau_ngay_chot`: dòng ngày 08/10 được nhận diện và loại khỏi kỳ chốt 07/10. Không thay ngày lỗi bằng ngày hiện tại và không biến giá thiếu thành 0.

Chạy cell dưới để xem trực tiếp sự khác biệt giữa hai cách parse. Nếu nguồn không trả dòng có giờ tại lúc bạn chạy, kết quả rỗng là hợp lệ. Với báo cáo cuối ngày, hãy chọn một ngày đã đóng cửa; một dòng trong phiên có cùng DATE vẫn chưa phải giá đóng cửa chính thức.

In [22]:
-- SQL 11b | Dòng trong phiên có cả ngày và giờ
SELECT ma_nganh, s AS ngay_goc, i AS chi_so_goc,
       COALESCE(TRY_STRPTIME(s,'%d/%m/%y'),
                TRY_STRPTIME(s,'%d/%m/%Y'))::DATE AS neu_chi_doc_ngay,
       COALESCE(TRY_STRPTIME(s,'%d/%m/%y %H:%M'),
                TRY_STRPTIME(s,'%d/%m/%Y %H:%M'))::DATE AS ngay_sau_chuan_hoa,
       CASE WHEN COALESCE(TRY_STRPTIME(s,'%d/%m/%y %H:%M'),
                         TRY_STRPTIME(s,'%d/%m/%Y %H:%M'))::DATE > getvariable('tut_as_of')
            THEN 'Ngoài ngày chốt: không dùng cho kỳ này'
            ELSE 'Kiểm tra phiên đã đóng cửa chưa' END AS xu_ly
FROM tut_industry_raw
WHERE contains(s, ':')
ORDER BY ma_nganh;

ma_nganh,ngay_goc,chi_so_goc,neu_chi_doc_ngay,ngay_sau_chuan_hoa,xu_ly
0500,08/10/26 09:33,109.09,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
1300,08/10/26 09:33,281.16,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
1700,08/10/26 09:33,208.83,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
2300,08/10/26 09:33,238.22,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
2700,08/10/26 09:33,533.98,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
3300,08/10/26 09:33,90.24,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
3500,08/10/26 09:33,157.58,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
3700,08/10/26 09:33,181.12,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
4500,08/10/26 09:33,151.92,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này
5300,08/10/26 09:33,142.37,NaT,2026-10-08,Ngoài ngày chốt: không dùng cho kỳ này


## 12 · JOIN + PARTITION BY: xếp hạng ngành
**Câu hỏi:** ngành nào tăng mạnh hoặc giảm mạnh trong cùng khoảng 20 phiên?

`PARTITION BY ma_nganh` tách phép tính LAG cho từng ngành; thiếu phần này sẽ lấy nhầm giá của ngành khác. `LEFT JOIN` bổ sung tên ngành và giữ các ngành chưa có dữ liệu.

Chỉ gán thứ hạng khi đủ ít nhất 21 giá, không lỗi/trùng ngày và ngày cuối trùng ngày chốt. `DENSE_RANK` cho cùng hạng nếu lợi suất bằng nhau; xếp theo số chưa làm tròn. Cột `kiem_dinh` cho biết vì sao một ngành không đủ điều kiện xếp hạng.

**Đối chiếu kỳ mẫu:** Bảo hiểm +9,10%; Dầu khí +8,45%; Bán lẻ +6,30%. Hàng cá nhân & gia dụng −15,65%. Có 7/19 ngành dương. Đây là diễn biến chỉ số ngành, không phải lợi nhuận chắc chắn của mọi cổ phiếu trong ngành.

In [23]:
-- SQL 12 | Chuẩn hóa cùng kỳ, sau đó xếp hạng
CREATE OR REPLACE TEMP TABLE tut_industry_rank AS
WITH audit AS (
 SELECT ma_nganh,
        COUNT(*) FILTER (WHERE ngay IS NULL OR chi_so_nganh IS NULL OR chi_so_nganh<=0) AS loi_du_lieu,
        COUNT(ngay)-COUNT(DISTINCT ngay) AS ngay_trung
 FROM tut_industry_prices GROUP BY ma_nganh
),
metrics AS (
 SELECT *,
        LAG(chi_so_nganh,5) OVER w AS gia_truoc_5,
        LAG(chi_so_nganh,20) OVER w AS gia_truoc_20,
        COUNT(*) OVER (PARTITION BY ma_nganh) AS so_phien
 FROM tut_industry_prices
 WHERE ngay<=getvariable('tut_as_of') AND chi_so_nganh>0
 WINDOW w AS (PARTITION BY ma_nganh ORDER BY ngay)
),
latest AS (
 SELECT * FROM metrics
 QUALIFY ROW_NUMBER() OVER (PARTITION BY ma_nganh ORDER BY ngay DESC)=1
),
joined AS (
 SELECT c.ma_nganh,c.ten_nganh,l.ngay,l.so_phien,
        l.chi_so_nganh, l.gia_truoc_5,l.gia_truoc_20,
        a.loi_du_lieu,a.ngay_trung,
        CASE WHEN l.ngay IS NULL THEN 'Thiếu dữ liệu'
             WHEN a.loi_du_lieu>0 OR a.ngay_trung>0 THEN 'Kiểm tra nguồn'
             WHEN l.ngay<>getvariable('tut_as_of') THEN 'Khác ngày chốt'
             WHEN l.gia_truoc_20 IS NULL THEN 'Thiếu 21 giá'
             ELSE 'Đúng kỳ' END AS kiem_dinh
 FROM tut_industry_catalog c LEFT JOIN latest l USING(ma_nganh)
 LEFT JOIN audit a USING(ma_nganh)
),
returns AS (
 SELECT *,
        CASE WHEN kiem_dinh='Đúng kỳ'
             THEN 100*(chi_so_nganh/NULLIF(gia_truoc_20,0)-1) END AS r20,
        CASE WHEN kiem_dinh='Đúng kỳ'
             THEN 100*(chi_so_nganh/NULLIF(gia_truoc_5,0)-1) END AS r5
 FROM joined
)
SELECT CASE WHEN r20 IS NOT NULL
            THEN DENSE_RANK() OVER (ORDER BY r20 DESC NULLS LAST) END AS hang,
       ma_nganh,ten_nganh,ngay,
       ROUND(r5,2) AS loi_suat_5_phien_pct,
       ROUND(r20,2) AS loi_suat_20_phien_pct,
       so_phien,loi_du_lieu,ngay_trung,kiem_dinh
FROM returns;

SELECT * FROM tut_industry_rank ORDER BY hang NULLS LAST,ma_nganh;

hang,ma_nganh,ten_nganh,ngay,loi_suat_5_phien_pct,loi_suat_20_phien_pct,so_phien,loi_du_lieu,ngay_trung,kiem_dinh
1,8500,Bảo hiểm,2026-10-07,0.29,9.10,248,0,0,Đúng kỳ
2,0500,Dầu khí,2026-10-07,0.62,8.45,248,0,0,Đúng kỳ
3,5300,Bán lẻ,2026-10-07,2.89,6.30,248,0,0,Đúng kỳ
4,2700,Hàng & Dịch vụ Công nghiệp,2026-10-07,2.08,3.92,248,0,0,Đúng kỳ
5,1300,Hóa chất,2026-10-07,0.55,3.28,248,0,0,Đúng kỳ
6,1700,Tài nguyên Cơ bản,2026-10-07,4.94,2.17,248,0,0,Đúng kỳ
7,4500,Y tế,2026-10-07,1.14,0.66,248,0,0,Đúng kỳ
8,3500,Thực phẩm và đồ uống,2026-10-07,2.02,-0.23,248,0,0,Đúng kỳ
9,8300,Ngân hàng,2026-10-07,-1.82,-2.20,248,0,0,Đúng kỳ
10,2300,Xây dựng và Vật liệu,2026-10-07,-1.04,-2.84,248,0,0,Đúng kỳ


## 13 · Một SQL cell, hai cách xem: Table và Visualize
**Câu hỏi:** sự phân hóa giữa các ngành trông như thế nào?

Cell bên dưới đọc kết quả đã tính, kèm cấu hình **GGSQL** để có bảng và biểu đồ cột trong cùng output. Mở phần **Visualize** để xem biểu đồ; phần **Table** giữ số chính xác và tên đầy đủ.

Trục X là mã ngành để 19 nhãn dễ đọc; tra tên trong bảng cùng cell. Trục Y là lợi suất 20 phiên (%). Cột dưới 0 là giảm giá. Thứ tự biểu đồ theo mã ngành; bảng xếp hạng ở bài 12 theo hiệu suất.

Cấu hình biểu đồ của cell này:
```text
ma_nganh AS x, loi_suat_20_phien_pct AS y
DRAW bar SETTING position => 'identity'
LABEL title => 'Lợi suất 20 phiên theo ngành (%)'
```

Đây là cấu hình phần hiển thị GGSQL, tách khỏi câu SELECT. Không dán riêng đoạn cấu hình này vào một SQL cell thông thường. Những ngành thiếu điều kiện bị loại khỏi biểu đồ bằng `WHERE kiem_dinh='Đúng kỳ'`, nhưng vẫn được giữ ở bảng kiểm tra và bảng xếp hạng phía trên.

Với lợi suất có cả số âm, dùng `SETTING position => 'identity'` để vẽ cột tại đúng giá trị thay vì xếp chồng. [Tài liệu vị trí cột GGSQL](https://ggsql.org/syntax/layer/position/identity.html).

In [25]:
-- SQL 13 | SELECT cho cả bảng và biểu đồ
SELECT ma_nganh,ten_nganh,ngay,loi_suat_20_phien_pct
FROM tut_industry_rank
WHERE kiem_dinh='Đúng kỳ'
ORDER BY ma_nganh;

ma_nganh,ten_nganh,ngay,loi_suat_20_phien_pct
0500,Dầu khí,2026-10-07,8.45
1300,Hóa chất,2026-10-07,3.28
1700,Tài nguyên Cơ bản,2026-10-07,2.17
2300,Xây dựng và Vật liệu,2026-10-07,-2.84
2700,Hàng & Dịch vụ Công nghiệp,2026-10-07,3.92
3300,Ô tô và phụ tùng,2026-10-07,-5.50
3500,Thực phẩm và đồ uống,2026-10-07,-0.23
3700,Hàng cá nhân & Gia dụng,2026-10-07,-15.65
4500,Y tế,2026-10-07,0.66
5300,Bán lẻ,2026-10-07,6.30


## 14 · Kiểm tra trước khi dùng kết quả
**Câu hỏi:** các phần của báo cáo có đang dùng đúng cấu hình và cùng ngày không?

Cell dưới kiểm tra cấu hình, số quan sát, độ phủ và ngày dữ liệu của các bảng vừa tạo. **Đạt** nghĩa là qua các kiểm tra được liệt kê, không phải chứng nhận dữ liệu nguồn hoàn toàn đúng. Đặc biệt, nó chưa đối chiếu lịch nghỉ sàn và chưa xác minh phương pháp tính chỉ số ngành.

Nếu thấy **Cần xem lại**, trở lại bài tương ứng; không đổi điều kiện kiểm tra chỉ để nhận nhãn Đạt. Sau khi sửa tham số tại SQL 03, cách an toàn là chạy lại toàn bộ SQL phía sau theo thứ tự.

Các điểm khác nhau giữa nguồn cần giữ riêng: giá trị `accValue` chưa chắc cùng phạm vi với tổng thanh khoản báo chí; số khối ngoại có thể khác do phạm vi/thời điểm cập nhật. Tutorial ưu tiên truy vết kết quả về đúng bảng nguồn.

In [26]:
-- SQL 14 | Kiểm tra tổng hợp các bảng trong session
WITH checks AS (
 SELECT 1 AS thu_tu, 'Cấu hình giá khớp cell tham số' AS phep_kiem_tra,
        MIN(requested_as_of)=getvariable('tut_as_of')
          AND MIN(requested_ticker)=getvariable('tut_ticker') AS dat,
        COUNT(*)::VARCHAR || ' dòng giá đã tải' AS chi_tiet
 FROM tut_prices_raw
 UNION ALL
 SELECT 2,'Lịch sử đủ MA50 và không trùng ngày',
        COUNT(*)>=50 AND COUNT(*)=COUNT(DISTINCT ngay),
        COUNT(*)::VARCHAR || ' phiên trong kỳ'
 FROM tut_prices
 UNION ALL
 SELECT 3,'Chỉ số kết thúc đúng ngày chốt',
        MAX(ngay)=getvariable('tut_as_of'), MAX(ngay)::VARCHAR
 FROM tut_metrics
 UNION ALL
 SELECT 4,'Độ rộng đủ 3 sàn và đúng kỳ',
        COUNT(*)=3 AND BOOL_AND(kiem_dinh='Đúng kỳ'),
        COUNT(*) FILTER (WHERE kiem_dinh='Đúng kỳ')::VARCHAR || '/3 sàn đạt'
 FROM tut_breadth
 UNION ALL
 SELECT 5,'Khối ngoại đủ 3 sàn, đủ 20 phiên',
        COUNT(*)=3 AND BOOL_AND(kiem_dinh='Đúng kỳ'),
        COUNT(*) FILTER (WHERE kiem_dinh='Đúng kỳ')::VARCHAR || '/3 sàn đạt'
 FROM tut_foreign
 UNION ALL
 SELECT 6,'Thanh khoản đúng ngày và đủ cửa sổ',
        ngay=getvariable('tut_as_of') AND n20=20
          AND gia_tri>=0 AND tb20_truoc>0,
        ngay::VARCHAR || '; ' || n20::VARCHAR || ' phiên trước'
 FROM (SELECT * FROM tut_liquidity ORDER BY ngay DESC LIMIT 1)
 UNION ALL
 SELECT 7,'Toàn bộ ngành đủ điều kiện xếp hạng',
        COUNT(*)=(SELECT COUNT(*) FROM tut_industry_catalog)
          AND BOOL_AND(kiem_dinh='Đúng kỳ'),
        COUNT(*) FILTER (WHERE kiem_dinh='Đúng kỳ')::VARCHAR || '/' ||
          COUNT(*)::VARCHAR || ' ngành đạt'
 FROM tut_industry_rank
)
SELECT phep_kiem_tra,
       CASE WHEN COALESCE(dat,FALSE) THEN 'Đạt' ELSE 'Cần xem lại' END AS ket_qua,
       chi_tiet
FROM checks ORDER BY thu_tu;

phep_kiem_tra,ket_qua,chi_tiet
Cấu hình giá khớp cell tham số,Đạt,250 dòng giá đã tải
Lịch sử đủ MA50 và không trùng ngày,Đạt,250 phiên trong kỳ
Chỉ số kết thúc đúng ngày chốt,Đạt,2026-10-07
Độ rộng đủ 3 sàn và đúng kỳ,Đạt,3/3 sàn đạt
"Khối ngoại đủ 3 sàn, đủ 20 phiên",Đạt,3/3 sàn đạt
Thanh khoản đúng ngày và đủ cửa sổ,Đạt,2026-10-07; 20 phiên trước
Toàn bộ ngành đủ điều kiện xếp hạng,Đạt,19/19 ngành đạt


## 15 · Bài tập tự thực hành
Tạo thêm SQL cell sau bài học để thử, giữ các cell mẫu làm bản tham chiếu.

1. **Lọc ngành:** từ `tut_industry_rank`, tìm ngành có lợi suất 20 phiên dương. Với cấu hình mẫu có **7 ngành**; dùng `WHERE loi_suat_20_phien_pct > 0`, kiểm tra `kiem_dinh`.
2. **Thêm MA200:** mở rộng SQL 06 bằng cửa sổ 199 phiên trước + hiện tại; chỉ xuất khi đủ 200 quan sát. Kỳ mẫu VNINDEX có MA200 khoảng **1.796,65**.
3. **Đổi chỉ số:** đặt `tut_ticker='VN30'`, chạy lại từ SQL 03. So sánh giá với MA20 và lợi suất 20 phiên. Sau đó khôi phục VNINDEX nếu muốn khớp các mốc mẫu.
4. **Mạnh tương đối hay tăng tuyệt đối?** tìm ngành vượt VNINDEX trong 20 phiên nhưng lợi suất ngành vẫn âm. Gợi ý lời giải ở SQL 15 bên dưới.
5. **Viết một đoạn phân tích:** dùng một Markdown cell viết 5 câu: ngày chốt → xu hướng chỉ số → thanh khoản → khối ngoại → ngành và hạn chế dữ liệu. Mỗi số phải truy được về một output.

### Lời giải mẫu cho bài 4

`chenh_lech_diem_phan_tram` = lợi suất ngành − lợi suất chỉ số tham chiếu. Ví dụ −2% so với −4% là vượt 2 **điểm phần trăm**, dù vẫn giảm 2%. Đây là phép so hiệu suất giá đơn giản, chưa điều chỉnh rủi ro.

Lời giải dùng `CROSS JOIN` với đúng một dòng chỉ số tham chiếu; đồng thời kiểm tra **cùng ngày đầu và ngày cuối** của cửa sổ 20 phiên. Nếu đổi mã tham chiếu ở SQL 03, tên chỉ số ở kết quả sẽ đổi theo.

**Kỳ mẫu VNINDEX:** 4 ngành giảm nhưng tốt hơn chỉ số: Thực phẩm và đồ uống, Ngân hàng, Xây dựng và Vật liệu, Điện/nước/xăng dầu khí đốt.

In [27]:
-- SQL 15 | Giảm giá nhưng tốt hơn chỉ số tham chiếu
WITH benchmark AS (
 SELECT chi_so,ngay,
        LAG(ngay,20) OVER w AS ngay_dau,
        ROUND(100*(close/NULLIF(LAG(close,20) OVER w,0)-1),2) AS loi_suat_chi_so_pct
 FROM tut_prices
 WINDOW w AS (ORDER BY ngay)
 QUALIFY ROW_NUMBER() OVER (ORDER BY ngay DESC)=1
),
sector_window AS (
 SELECT ma_nganh,ngay,
        LAG(ngay,20) OVER (PARTITION BY ma_nganh ORDER BY ngay) AS ngay_dau
 FROM tut_industry_prices WHERE ngay<=getvariable('tut_as_of')
 QUALIFY ROW_NUMBER() OVER (PARTITION BY ma_nganh ORDER BY ngay DESC)=1
)
SELECT s.ma_nganh,s.ten_nganh,b.chi_so AS tham_chieu,
       b.ngay_dau,b.ngay AS ngay_cuoi,
       s.loi_suat_20_phien_pct AS loi_suat_nganh_pct,
       b.loi_suat_chi_so_pct,
       ROUND(s.loi_suat_20_phien_pct-b.loi_suat_chi_so_pct,2)
         AS chenh_lech_diem_phan_tram
FROM tut_industry_rank s
JOIN sector_window w USING(ma_nganh)
CROSS JOIN benchmark b
WHERE s.kiem_dinh='Đúng kỳ'
  AND w.ngay=b.ngay AND w.ngay_dau=b.ngay_dau
  AND s.loi_suat_20_phien_pct<0
  AND s.loi_suat_20_phien_pct>b.loi_suat_chi_so_pct
ORDER BY chenh_lech_diem_phan_tram DESC;

ma_nganh,ten_nganh,tham_chieu,ngay_dau,ngay_cuoi,loi_suat_nganh_pct,loi_suat_chi_so_pct,chenh_lech_diem_phan_tram
3500,Thực phẩm và đồ uống,VNINDEX,2026-09-09,2026-10-07,-0.23,-4.04,3.81
8300,Ngân hàng,VNINDEX,2026-09-09,2026-10-07,-2.20,-4.04,1.84
2300,Xây dựng và Vật liệu,VNINDEX,2026-09-09,2026-10-07,-2.84,-4.04,1.20
7500,"Điện, nước & xăng dầu khí đốt",VNINDEX,2026-09-09,2026-10-07,-2.93,-4.04,1.11


## 16 · Tự xử lý lỗi và dùng lại quy trình

| Hiện tượng | Cách kiểm tra và xử lý |
|---|---|
| Không thấy `tcinvest` trong catalog | Gắn global connection có sẵn vào đúng notebook; kiểm tra trạng thái kết nối. |
| Không tìm thấy view/hàm `tcinvest_...` | Chạy cell datasource setup; xem lại invoke SQL trong catalog, phân biệt view và hàm có đối số. |
| `tut_prices_raw` hoặc bảng tạm khác không tồn tại | Kernel có thể đã khởi động lại hoặc bạn chạy sai thứ tự. Chạy từ SQL 03 và các cell tải nguồn trở xuống. |
| Biến cấu hình trả NULL | Chạy SQL 03; biến chỉ tồn tại trong session. |
| Không có dòng hoặc ngày cuối cũ | Xem ngày chốt, mã/sàn/ngành, số phiên yêu cầu và trạng thái API; đối chiếu lịch giao dịch. |
| Parse ngày ra NULL | Xem chuỗi gốc. Nguồn có thể trộn `dd/mm/yy` với `dd/mm/yy HH:MM`; bài 11b là ví dụ trực tiếp. |
| Lợi suất/MA là NULL | Kiểm tra độ dài cửa sổ, giá trước đó và mẫu số; không thay bằng 0. |
| API timeout hoặc lỗi kết nối | Kiểm tra dịch vụ của global connection đang chạy; chạy lại cell tải nguồn bị lỗi rồi những cell phụ thuộc. |
| Bảng có dữ liệu nhưng thiếu biểu đồ | Xem stderr, kiểm tra kết quả SQL không rỗng và cấu hình GGSQL. Cột có số âm dùng `position='identity'`. |
| Sửa ngày mà output vẫn cũ | Output chỉ đổi sau khi chạy. Chạy lại chuỗi cell tạo bảng tạm và các cell tính kết quả. |

### Ghi nhớ khi tái sử dụng

**Global connection** được dùng chung; **bảng tạm và biến SQL** thuộc session; **output notebook** là kết quả của lần chạy đã lưu. Ba lớp này có vòng đời khác nhau.

`WITH ...` đặt tên các bước trong **một câu truy vấn**. `CREATE OR REPLACE TEMP TABLE ...` giữ kết quả cho **nhiều cell** trong cùng session. Tutorial chủ động dùng bảng tạm để tách tải nguồn khỏi phân tích; những cell này chạy tuần tự, chưa khai báo DAG tự động.

Để chuyển sang mã cổ phiếu, trước hết xem lại schema và tham số `type` trong catalog; không chỉ đổi `VNINDEX` thành mã cổ phiếu trong ví dụ `type='index'`. Kiểm tra đơn vị giá, điều chỉnh cổ tức/chia tách và kỳ dữ liệu trước khi so sánh.

### Mẫu đoạn nhận định để tự viết

> Với dữ liệu chốt ngày **[ngày từ output]**, **[chỉ số]** có lợi suất 20 phiên **[x%]** và đang **[trên/dưới]** MA20/MA50. Giá trị giao dịch HOSE bằng **[x lần]** trung bình 20 phiên trước. Khối ngoại trên HOSE mua/bán ròng **[x tỷ đồng]** trong 20 phiên. Các nhóm ngành mạnh/yếu là **[tên ngành và lợi suất]**. Kết luận này dựa trên giá và dòng tiền; cần bổ sung **[dữ liệu còn thiếu]** trước khi đánh giá định giá hoặc chọn cổ phiếu.

Mốc số trong phần giải thích là kết quả ví dụ ngày 07/10/2026, **không tự thay đổi khi bạn chạy lại**. Luôn lấy số trong output mới nhất để viết nhận định. Bài học chưa đánh giá P/E, lợi nhuận doanh nghiệp hoặc danh mục cá nhân.

### Tài liệu tham khảo

- Nguồn và tham số TCInvest: catalog **Connections → tcinvest**, cùng SQL 02 trong notebook này.
- [DuckDB: biến SQL](https://www.duckdb.org/docs/current/sql/statements/set_variable).
- [DuckDB: window functions](https://www.duckdb.org/docs/current/sql/functions/window_functions).
- [GGSQL: cấu hình biểu đồ cột](https://ggsql.org/syntax/layer/type/bar.html).

**Hoàn thành bài học khi:** bạn tự đổi được ngày/mã tham chiếu, chạy lại các cell, giải thích được NULL và ngày dữ liệu, đọc được các kiểm tra ở SQL 14, và viết được đoạn nhận định có dẫn số từ output.